# CCRCB Profiling Exploration
Load data using `ccrcb_analysis` and plot metrics interactively.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
sys.path.append('..') # Add project root to path

import pandas as pd
import matplotlib.pyplot as plt
import ccrcb_analysis.parsers as parsers
import ccrcb_analysis.plotters as plotters

# Ensure plots display in notebook
%matplotlib inline

## 1. Load Data
You can load data from a single run directory, or pass a **list of directories** to seamlessly merge pure timings with hardware metrics, or combine runs across different scales.

In [ ]:
# Define the directories you want to load and merge
# The parser uses the folder names (_timing_ or _metrics_) to auto-detect what it is reading.
dir_timing = "../results_timing_2_ranks_20260422_065504"
dir_metrics = "../results_metrics_2_ranks_20260422_084316"

# Load data by passing a list of folders
df = parsers.load_run_data([dir_timing, dir_metrics])

if not df.empty:
    print(f"Has Timing Data: {df['Has_Timing_Data'].any()}")
    print(f"Has GPU Metrics: {df['Has_GPU_Metrics'].any()}")
    display(df.head())
else:
    print("No data loaded. Check if the directories exist.")

## 2. Plotting Timing Heatmaps
Compare Message Size vs Math Load for timing metrics like Serial Speedup.

In [ ]:
if not df.empty and df['Has_Timing_Data'].any():
    backends = df['Backend'].unique()
    fig, axes = plt.subplots(1, len(backends), figsize=(15, 5), sharey=True)
    if len(backends) == 1: axes = [axes]
    
    for ax, backend in zip(axes, backends):
        sub_df = df[df['Backend'] == backend]
        if len(sub_df) > 0:
            plotters.plot_heatmap(sub_df, metric="Serial_Speedup", 
                                  title=f"{backend} - Serial Speedup", ax=ax)
    
    plt.tight_layout()
    plt.show()
else:
    print("Timing data not available in this dataset. Skipping timing plots.")

## 3. Plotting Hardware Metrics

In [ ]:
if not df.empty and df['Has_GPU_Metrics'].any():
    fig, ax = plt.subplots(figsize=(10, 6))
    plotters.plot_contention_scatter(df, x_metric="Avg_DRAM_Read_Bandwidth_[BW_%]", y_metric="Contention_Factor", hue="Backend", ax=ax)
    plt.show()
else:
    print("Hardware metrics not available in this dataset. Skipping contention scatter plot.")

## 4. Plotting Heatmaps
Compare Message Size vs Math Load for different metrics.

In [ ]:
if not df.empty:
    backends = df['Backend'].unique()
    fig, axes = plt.subplots(1, len(backends), figsize=(15, 5), sharey=True)
    if len(backends) == 1: axes = [axes]
    
    for ax, backend in zip(axes, backends):
        sub_df = df[df['Backend'] == backend]
        if len(sub_df) > 0:
            plotters.plot_heatmap(sub_df, metric="Overlap_Efficiency", 
                                  title=f"{backend} - Overlap Efficiency", ax=ax)
    
    plt.tight_layout()
    plt.show()

    fig, axes = plt.subplots(1, len(backends), figsize=(15, 5), sharey=True)
    if len(backends) == 1: axes = [axes]
    
    for ax, backend in zip(axes, backends):
        sub_df = df[df['Backend'] == backend]
        if len(sub_df) > 0:
            plotters.plot_heatmap(sub_df, metric="Contention_Factor", 
                                  title=f"{backend} - Contention Factor (Loss L)", cmap="magma_r", ax=ax)
    
    plt.tight_layout()
    plt.show()
